In [ ]:
# Berlin Food Delivery ETA Prediction
## Hyperparameter Tuning

Goal: Tune Gradient Boosting to push past the current R² ≈ 0.677 ceiling from 
default parameters, using RandomizedSearchCV for efficiency.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

In [2]:
## reloaded data and recreated the same train/val split

train = pd.read_csv('train_features.csv')

drop_cols = [
    'ID', 'Delivery_person_ID', 'Order_Date', 'Time_Orderd', 'Time_Order_picked',
    'Restaurant_latitude', 'Restaurant_longitude', 
    'Delivery_location_latitude', 'Delivery_location_longitude',
    'Weatherconditions', 'Road_traffic_density'
]

X = train.drop(columns=drop_cols + ['Time_taken(min)'])
y = train['Time_taken(min)']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_val.shape)

(36474, 25) (9119, 25)


In [3]:
## defined the parameter search space

param_distributions = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'min_samples_split': [2, 5, 10]
}

In [4]:
## ran RandomizedSearchCV

gb_base = GradientBoostingRegressor(random_state=42)

random_search = RandomizedSearchCV(
    estimator=gb_base,
    param_distributions=param_distributions,
    n_iter=25,           # number of random combinations to try
    scoring='r2',
    cv=3,                # 3-fold cross-validation
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start = time.time()
random_search.fit(X_train, y_train)
elapsed = time.time() - start

print(f"Search completed in {elapsed/60:.1f} minutes")
print("Best params:", random_search.best_params_)
print("Best CV R²:", random_search.best_score_)

Fitting 3 folds for each of 25 candidates, totalling 75 fits
Search completed in 2.3 minutes
Best params: {'subsample': 0.7, 'n_estimators': 100, 'min_samples_split': 2, 'max_depth': 6, 'learning_rate': 0.05}
Best CV R²: 0.6710477999027774


In [5]:
## evaluate tuned model on validation set

best_gb = random_search.best_estimator_

val_preds = best_gb.predict(X_val)
mae = mean_absolute_error(y_val, val_preds)
rmse = np.sqrt(mean_squared_error(y_val, val_preds))
r2 = r2_score(y_val, val_preds)

print(f"Tuned Gradient Boosting -> MAE: {mae:.3f} | RMSE: {rmse:.3f} | R²: {r2:.3f}")

Tuned Gradient Boosting -> MAE: 4.178 | RMSE: 5.287 | R²: 0.681


In [6]:
##  before vs after comparison

comparison = pd.DataFrame([
    {'model': 'Gradient Boosting (default)', 'MAE': 4.216, 'RMSE': 5.323, 'R2': 0.677},
    {'model': 'Gradient Boosting (tuned)', 'MAE': mae, 'RMSE': rmse, 'R2': r2}
])
comparison

,model,MAE,RMSE,R2
0,Gradient Boosting (default),4.216000,5.32300,0.677000
1,Gradient Boosting (tuned),4.177817,5.28651,0.681252


In [7]:
train_preds_tuned = best_gb.predict(X_train)
train_r2_tuned = r2_score(y_train, train_preds_tuned)
train_mae_tuned = mean_absolute_error(y_train, train_preds_tuned)
print(f"Tuned GB (train) -> MAE: {train_mae_tuned:.3f} | R²: {train_r2_tuned:.3f}")

Tuned GB (train) -> MAE: 4.110 | R²: 0.691


In [ ]:
## Summary: Hyperparameter Tuning Complete

- Used RandomizedSearchCV (25 candidates × 3-fold CV = 75 fits) to tune Gradient Boosting 
  across n_estimators, max_depth, learning_rate, subsample, and min_samples_split.
- **Best parameters found:** `n_estimators=100, max_depth=6, learning_rate=0.05, 
  subsample=0.7, min_samples_split=2`
- **Result:** MAE improved from 4.216 → 4.178 min, R² improved from 0.677 → 0.681 — 
  a modest but genuine improvement over the default model.
- Confirmed no overfitting: train R² (0.691) vs. validation R² (0.681), a gap of only 
  0.010 — slightly tighter than the default model's gap, despite using a deeper tree depth.
- Tuning yielded incremental rather than dramatic gains, which is expected: Gradient 
  Boosting's default hyperparameters were already reasonably well-suited to this data.

### Final Model Comparison (All Steps)

| Model | MAE | RMSE | R² |
|---|---|---|---|
| Linear Regression | 5.103 | 6.443 | 0.527 |
| Decision Tree | 4.297 | 5.455 | 0.661 |
| Random Forest | 4.211 | 5.325 | 0.677 |
| Gradient Boosting (default) | 4.216 | 5.323 | 0.677 |
| **Gradient Boosting (tuned)** | **4.178** | **5.287** | **0.681** |

**Final selected model:** Tuned Gradient Boosting Regressor

In [8]:
## saved the final tuned model for reproducibility

import joblib
joblib.dump(best_gb, 'final_tuned_gb_model.pkl')
print("Model saved as final_tuned_gb_model.pkl")

Model saved as final_tuned_gb_model.pkl
